# Week 10 과제 · 모범답안 (교수자용)

> 교수자용 모범답안입니다. 학생에게 배포하지 마세요.

## 준비: 데이터와 자동 확인 도우미

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

DF = pd.read_csv(get_data("students.csv"), dtype={"student_id": str})
ORIGINAL = DF.copy()


def check(desc, test):
    """자동 확인 도우미: test는 True/False를 돌려주는 함수(lambda)입니다."""
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ 미작성 - {desc}")
    except Exception as e:
        print(f"❌ 오류 - {desc} ({type(e).__name__}: {e})")
    else:
        print(f"{'✅ 통과' if ok else '❌ 실패'} - {desc}")


def unchanged():
    """DF가 원래 상태 그대로인지."""
    return DF.equals(ORIGINAL)

print(DF.shape)

(80, 8)


---
## 문제 1. 데이터 개요 함수 (2점)

`overview(df)`는 다음 딕셔너리를 돌려줍니다.

| 키 | 값 |
|---|---|
| `"rows"`, `"cols"` | 행 수, 열 수 (정수) |
| `"missing"` | 결측이 **있는 열만** `{열 이름: 결측 개수}` |
| `"numeric"` | 숫자형 열 이름 리스트 (원래 열 순서) |
| `"categories"` | 문자열(object) 열마다 **고유값 개수** `{열 이름: 개수}` |

힌트: `df.isna().sum()`, `df.select_dtypes("number")`, `df.select_dtypes("object")`, `nunique()`

In [2]:
def overview(df):
    missing = df.isna().sum()
    return {
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1]),
        "missing": {col: int(n) for col, n in missing[missing > 0].items()},
        "numeric": list(df.select_dtypes("number").columns),
        "categories": {col: int(df[col].nunique()) for col in df.select_dtypes("object").columns},
    }

In [3]:
check("rows, cols", lambda: (overview(DF)["rows"], overview(DF)["cols"]) == (80, 8))
check("missing", lambda: overview(DF)["missing"] == {"assignment": 3})
check("numeric", lambda: overview(DF)["numeric"] == ["year", "midterm", "final", "assignment", "attendance"])
check("categories", lambda: overview(DF)["categories"] == {"student_id": 80, "name": 80, "department": 5})
check("결측 없는 표", lambda: overview(pd.DataFrame({"a": [1, 2]}))["missing"] == {})

✅ 통과 - rows, cols
✅ 통과 - missing
✅ 통과 - numeric
✅ 통과 - categories
✅ 통과 - 결측 없는 표


---
## 문제 2. 조건 필터와 정렬 (2점)

`find_students(df)`는 다음 조건의 학생을 돌려줍니다.

- 학과가 **컴퓨터공학과 또는 정보보안학과**
- **3학년 이상**
- 기말 **80점 이상**

결과는 `student_id, name, department, year, final` 열만, **기말 내림차순 → 같으면 학번 오름차순**으로 정렬하고, 인덱스는 0부터 다시 매깁니다 (`reset_index(drop=True)`).

In [4]:
def find_students(df):
    cond = (df["department"].isin(["컴퓨터공학과", "정보보안학과"])
            & (df["year"] >= 3)
            & (df["final"] >= 80))
    cols = ["student_id", "name", "department", "year", "final"]
    return (df.loc[cond, cols]
              .sort_values(["final", "student_id"], ascending=[False, True])
              .reset_index(drop=True))

In [5]:
EXPECTED_N, EXPECTED_FIRST = 8, "한현우"      # students.csv 기준값
check("열 구성", lambda: list(find_students(DF).columns) == ["student_id", "name", "department", "year", "final"])
check("인원", lambda: len(find_students(DF)) == EXPECTED_N)
check("조건을 모두 만족", lambda: ((find_students(DF)["year"] >= 3) & (find_students(DF)["final"] >= 80)).all()
      and set(find_students(DF)["department"]) <= {"컴퓨터공학과", "정보보안학과"})
check("정렬 순서", lambda: find_students(DF)["final"].is_monotonic_decreasing)
check("인덱스 0부터", lambda: list(find_students(DF).index) == list(range(len(find_students(DF)))))
check("첫 학생", lambda: find_students(DF).loc[0, "name"] == EXPECTED_FIRST)
check("원본 불변", unchanged)

✅ 통과 - 열 구성
✅ 통과 - 인원
✅ 통과 - 조건을 모두 만족
✅ 통과 - 정렬 순서
✅ 통과 - 인덱스 0부터
✅ 통과 - 첫 학생
✅ 통과 - 원본 불변


---
## 문제 3. 총점·등급 열 추가 (3점)

`add_total_grade(df)`는 원본을 바꾸지 않고, 다음 두 열을 추가한 **새 DataFrame**을 돌려줍니다. (8주차 중간 실기와 같은 규정)

- `total`: 중간 30% + 기말 40% + 과제 30% (과제 결측은 0점), 소수 첫째 자리 반올림
- `grade`: 90 이상 A, 80 이상 B, 70 이상 C, 60 이상 D, 그 외 F. **출석 12회 미만은 F**. 자료형은 문자열(object)이어도, category여도 됩니다.

그리고 `top_names(df, n)`은 `add_total_grade`를 이용해 총점 상위 n명의 이름 리스트를 돌려줍니다 (동점이면 학번 오름차순).

In [6]:
def add_total_grade(df):
    out = df.copy()
    out["total"] = (out["midterm"] * 0.3 + out["final"] * 0.4 + out["assignment"].fillna(0) * 0.3).round(1)
    grade = np.select(
        [out["total"] >= 90, out["total"] >= 80, out["total"] >= 70, out["total"] >= 60],
        ["A", "B", "C", "D"], default="F")
    out["grade"] = np.where(out["attendance"] < 12, "F", grade)
    return out


def top_names(df, n):
    t = add_total_grade(df)
    return t.sort_values(["total", "student_id"], ascending=[False, True])["name"].head(n).tolist()

In [7]:
check("열 추가", lambda: {"total", "grade"} <= set(add_total_grade(DF).columns))
check("원본 불변 (열이 늘지 않음)", lambda: (add_total_grade(DF), unchanged())[1])
check("등급 분포 = 8주차 결과", lambda: add_total_grade(DF)["grade"].astype(str).value_counts().to_dict()
      == {"A": 8, "B": 22, "C": 23, "D": 12, "F": 15})
check("총점 평균", lambda: round(add_total_grade(DF)["total"].mean(), 1) == 73.8)
tiny = pd.DataFrame({"student_id": ["1", "2", "3"], "name": ["가", "나", "다"], "midterm": [90, 100, 59],
                     "final": [90, 100, 60], "assignment": [90, np.nan, 61], "attendance": [12, 16, 16]})
check("경계 90.0 → A, 과제 결측 → 0점", lambda: add_total_grade(tiny)["total"].tolist() == [90.0, 70.0, 60.0])
check("등급 (출석 경계 12 포함)", lambda: add_total_grade(tiny)["grade"].astype(str).tolist() == ["A", "C", "D"])
check("top_names(3)", lambda: top_names(DF, 3) == ["황서윤", "장예린", "한현우"])

✅ 통과 - 열 추가
✅ 통과 - 원본 불변 (열이 늘지 않음)
✅ 통과 - 등급 분포 = 8주차 결과
✅ 통과 - 총점 평균
✅ 통과 - 경계 90.0 → A, 과제 결측 → 0점
✅ 통과 - 등급 (출석 경계 12 포함)
✅ 통과 - top_names(3)


---
## 문제 4. 학과별 탐색 보고 (AI 협업, 3점)

`dept_report(df)`는 학과를 인덱스(가나다순)로 하는 DataFrame을 돌려줍니다.

| 열 | 의미 |
|---|---|
| `students` | 학생 수 (결측과 관계없이 전체 행 수) |
| `final_mean` | 기말 평균 (소수 첫째 자리) |
| `assign_missing` | 과제 결측 학생 수 |
| `low_attend` | 출석 12회 미만 학생 수 |
| `pass_rate` | 통과율(%): 문제 3의 등급이 F가 아닌 비율 × 100, 소수 첫째 자리 |

**AI 협업 과제**: AI에게 이 표를 요청한 뒤 다음을 확인하세요.
- `count()`와 `size()`의 차이 때문에 `students`가 결측만큼 적게 나오지 않는가?
- `students`의 합이 전체 행 수(80)와 같은가?

In [8]:
def dept_report(df):
    t = add_total_grade(df)
    t["assign_na"] = t["assignment"].isna()
    t["low"] = t["attendance"] < 12
    t["passed"] = t["grade"].astype(str) != "F"
    rep = t.groupby("department").agg(
        students=("student_id", "size"),
        final_mean=("final", "mean"),
        assign_missing=("assign_na", "sum"),
        low_attend=("low", "sum"),
        pass_rate=("passed", "mean"),
    ).sort_index()
    rep["final_mean"] = rep["final_mean"].round(1)
    rep["pass_rate"] = (rep["pass_rate"] * 100).round(1)
    return rep

In [9]:
EXPECTED_CS_ROW = [17, 73.6, 1, 1, 82.4]        # students.csv 기준값
check("열 구성", lambda: list(dept_report(DF).columns) == ["students", "final_mean", "assign_missing", "low_attend", "pass_rate"])
check("학과 가나다순 인덱스", lambda: list(dept_report(DF).index) == ["간호학과", "경영학과", "전자공학과", "정보보안학과", "컴퓨터공학과"])
check("students 합계 80 (size 사용)", lambda: dept_report(DF)["students"].sum() == 80)
check("assign_missing 합계 3", lambda: dept_report(DF)["assign_missing"].sum() == 3)
check("컴퓨터공학과 행", lambda: dept_report(DF).loc["컴퓨터공학과"].tolist() == EXPECTED_CS_ROW)
check("원본 불변", unchanged)

✅ 통과 - 열 구성
✅ 통과 - 학과 가나다순 인덱스
✅ 통과 - students 합계 80 (size 사용)
✅ 통과 - assign_missing 합계 3
✅ 통과 - 컴퓨터공학과 행
✅ 통과 - 원본 불변


### AI 활용 기록 (예시)

| 항목 | 내용 |
|---|---|
| 사용한 AI 도구 | (예) ChatGPT |
| count/size | `students=("assignment", "count")`를 써서 과제 결측 학생이 빠졌음 → 합계가 77로 나와 발견, `size`로 수정 |
| 체인 인덱싱 | 통과 여부 열을 `df[df.grade != "F"]["passed"] = True`로 만들어 값이 들어가지 않았음 → 불리언 열을 직접 계산 |
| 직접 고친 부분 | `add_total_grade`를 재사용해 원본을 바꾸지 않게 하고, 결측·출석 조건을 불리언 열로 만든 뒤 `sum`/`mean`으로 집계 |

---
## 성찰 (필수)

1. 8주차에 직접 구현했던 계산을 pandas로 다시 하면서 느낀 차이는?
2. 결측값을 0으로 채우는 것과 평균으로 채우는 것은 결론을 어떻게 바꿀 수 있을까요?
3. AI의 pandas 코드를 검증할 때 가장 먼저 확인하는 것은 무엇인가요?

(학생 작성 영역)